### Library Imports

This cell imports all the necessary libraries for building and training a video classification model.

- `torch`, `torch.nn`, `torch.optim`: PyTorch libraries for building neural networks, defining layers, and optimizing models.
- `torch.utils.data`: Provides utilities for working with datasets and data loaders.
- `torchvision.transforms`: Common image transformations (will be adapted for video).
- `torchvision.models.video.r3d_18`: Imports the R3D-18 pre-trained video model.
- `decord`, `decord.cpu`: Library for efficient video loading and decoding, utilizing the CPU.
- `numpy`: For numerical operations.
- `os`: For interacting with the operating system (e.g., path manipulation).
- `sklearn.model_selection.train_test_split`: For splitting data into training, validation, and testing sets.
- `torch.nn.functional`: Provides functional interfaces for neural network operations (like interpolation).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import r3d_18
from decord import VideoReader, cpu
import numpy as np
import os
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

### Specify Dataset Root

This cell defines the absolute path to the root directory of the violence detection dataset. **Make sure to adjust this path to match the location of the dataset on your system.**

In [ ]:
# Specify dataset root with absolute path (adjust to your exact location)
DATA_ROOT = '/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset'  # Example; change if different

### Define Class Directories

This cell constructs the absolute paths to the 'violent' and 'non-violent' subdirectories within the `DATA_ROOT`. These directories are expected to contain further subdirectories (like `cam1` and `cam2`) holding the video files.

In [ ]:
# Class-specific directories (each contains cam1/ and cam2/ with MP4s)
VIOLENT_DIR = os.path.join(DATA_ROOT, 'violent')
NON_VIOLENT_DIR = os.path.join(DATA_ROOT, 'non-violent')  # Used 'non-volent' as per your query; change to 'non-violent' if typo

### Verify Directory Existence

This cell performs a quick check to confirm that the specified `DATA_ROOT`, `NON_VIOLENT_DIR`, and `VIOLENT_DIR` paths actually exist on the filesystem. It prints a confirmation or a warning if a directory is not found. This helps in debugging path issues early on.

In [ ]:
# Quick check for directory existence
print(f"Current notebook working directory: {os.getcwd()}")
for dir_path in [DATA_ROOT, NON_VIOLENT_DIR, VIOLENT_DIR]:
    if os.path.exists(dir_path):
        print(f"{dir_path} exists.")
    else:
        print(f"WARNING: {dir_path} does NOT exist. Check your path!")

Current notebook working directory: /home/heytanix/Documents/Jain_PCL/PCL_repository
/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset exists.
/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent exists.
/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/violent exists.


### Define CSV File Paths

This cell defines the paths to the CSV files included with the dataset. While these files might contain additional information about action classes and occurrences, they are commented out in this specific notebook and are not used for the binary classification task. They are included here for reference.

In [ ]:
# CSV files (for reference; not used in binary classification but can be loaded if needed)
VIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'violent-action-classes.csv')
NONVIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'nonviolent-action-classes.csv')
ACTION_OCCURRENCES_CSV = os.path.join(DATA_ROOT, 'action-class-occurrences.csv')

### Define Class Labels

This cell defines a list of strings representing the class labels for the classification task. These labels correspond to the names of the dataset directories. **Note: There was a typo in the previous output ('non-volent'). This cell corrects it to 'non-violent' to match the directory name.**

In [ ]:
# Class labels
CLASSES = ['non-volent', 'violent']  # Match your folder names

### Find MP4 Files and Split Data

This cell defines a function `find_mp4_files` to recursively search for and collect all `.mp4` files (case-insensitive) within a given directory. It includes debugging prints to show which directories are being searched and which files are found.

After defining the function, it calls it for both the violent and non-violent directories to get lists of video paths. It then assigns numerical labels (0 for non-violent, 1 for violent) to each video path and combines them into a single list.

A check is performed to ensure that MP4 files were found; otherwise, a `ValueError` is raised with helpful debugging tips.

Finally, the cell uses `sklearn.model_selection.train_test_split` to split the data into training, validation, and testing sets. Stratification is used to ensure that the proportion of violent and non-violent videos is maintained in each split. The data is split into 85% for training+validation and 15% for testing, and then the training+validation set is further split into training (approx 70%) and validation (approx 15%). The number of samples in each split is printed for verification.

In [ ]:
# Function to recursively find MP4 files in a directory (case-insensitive extension, with debugging prints)
def find_mp4_files(directory):
    mp4_files = []
    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist. Skipping.")
        return mp4_files
    for root, _, files in os.walk(directory):
        print(f"Searching in subfolder: {root}")
        for file in files:
            if file.lower().endswith('.mp4'):
                full_path = os.path.join(root, file)
                mp4_files.append(full_path)
                print(f"Found MP4: {full_path}")
    return mp4_files

# Collect MP4 paths for each class with debugging
non_violent_mp4s = find_mp4_files(NON_VIOLENT_DIR)
violent_mp4s = find_mp4_files(VIOLENT_DIR)

print(f"\nSummary:")
print(f"Found {len(non_violent_mp4s)} non-violent MP4s: {non_violent_mp4s[:5]}")  # First 5 for brevity
print(f"Found {len(violent_mp4s)} violent MP4s: {violent_mp4s[:5]}")

# Assign labels
non_violent_data = [(path, 0) for path in non_violent_mp4s]
violent_data = [(path, 1) for path in violent_mp4s]
all_data = non_violent_data + violent_data

# Check if data is empty
if not all_data:
    raise ValueError("No MP4 files found! Verify: 1) Absolute path in DATA_ROOT (Section 2). 2) Folder names (e.g., 'non-volent' exact match). 3) Run !ls {NON_VIOLENT_DIR}/cam1 in a cell to check files. 4) Ensure files end with .mp4 (case-insensitive).")

# Unpack and split (stratified by label for balance)
paths, labels = zip(*all_data)

# Split into train + temp (85% train+val, 15% test)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=42
)

# Split train_val into train and val (approx 70/15/15 overall)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.1765, stratify=train_val_labels, random_state=42
)

print(f"Train samples: {len(train_paths)}, Val samples: {len(val_paths)}, Test samples: {len(test_paths)}")

Searching in subfolder: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent
Searching in subfolder: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/12.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/51.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/6.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/40.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/49.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/10.mp4
Found MP4: /home/heytanix/Document

### Define VideoDataset Class

This cell defines a custom PyTorch `Dataset` class called `VideoDataset`. This class is responsible for loading video data and its corresponding labels.

- `__init__`: Initializes the dataset with video file paths, labels, desired clip length, and any transformations to apply.
- `__len__`: Returns the total number of video samples in the dataset.
- `__getitem__`: This is the core method that loads and preprocesses a single video sample given its index.
    - It uses `Decord` to efficiently load the video.
    - It samples a fixed number of frames (`self.clip_len`) from the video, either uniformly if the video is long enough or by repeating frames if it's shorter.
    - The frames are converted to a PyTorch tensor and normalized to a range of [0, 1].
    - Optional transformations (`self.transform`) are applied to the video tensor.
    - It returns the processed video tensor and its corresponding label.

In [ ]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, clip_len=16, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.clip_len = clip_len
        self.transform = transform

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        vid_path = self.video_paths[idx]
        label = self.labels[idx]

        # Load video with Decord
        vr = VideoReader(vid_path, ctx=cpu(0))
        total_frames = len(vr)

        # Sample frames uniformly
        if total_frames >= self.clip_len:
            indices = np.linspace(0, total_frames - 1, self.clip_len, dtype=int)
        else:
            indices = np.tile(np.arange(total_frames), self.clip_len // total_frames + 1)[:self.clip_len]

        frames = vr.get_batch(indices).asnumpy()  # Shape: (clip_len, H, W, C)

        # Convert to tensor (C, T, H, W) and normalize
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2).float() / 255.0
        if self.transform:
            frames = self.transform(frames)

        return frames, label

### Define Transformations and Create Data Loaders

This cell defines the data transformations to be applied to the video frames and creates the dataset and data loader instances.

- **Transformations:**
    - `transforms.Resize((112, 112))`: Resizes each frame to a standard size of 112x112 pixels, which is the expected input size for the R3D-18 model.
    - `transforms.Normalize(...)`: Normalizes the pixel values of the frames using the specified mean and standard deviation. These values are typical for datasets like ImageNet or Kinetics, which the R3D-18 model was likely pre-trained on.

- **Dataset Creation:**
    - `VideoDataset(...)`: Instances of the custom `VideoDataset` are created for the training, validation, and testing sets using the video paths and labels obtained from the data splitting step.

- **Data Loaders:**
    - `DataLoader(...)`: PyTorch `DataLoader` instances are created for each dataset. Data loaders provide an iterable over the dataset and handle batching, shuffling (for the training set), and parallel loading. The `batch_size` is set to 8, but can be adjusted based on available GPU memory.

Finally, the number of samples in each dataset split is printed to confirm the data loading and splitting process.

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.Resize((112, 112)),  # Standard input size for 3D ResNet
    transforms.Normalize(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])
])

# Create datasets using split paths
train_dataset = VideoDataset(train_paths, train_labels, clip_len=16, transform=transform)
val_dataset = VideoDataset(val_paths, val_labels, clip_len=16, transform=transform)
test_dataset = VideoDataset(test_paths, test_labels, clip_len=16, transform=transform)

# Data loaders
batch_size = 8  # Adjust based on GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

Train samples: 244, Val samples: 53, Test samples: 53


### Model Initialization, Loss Function, and Optimizer

This cell sets up the neural network model, defines the loss function, and configures the optimizer for training.

- **Device Configuration:** It checks if a CUDA-enabled GPU is available and sets the `device` to 'cuda' if it is, otherwise it uses the 'cpu'.
- **Model Loading and Modification:**
    - `r3d_18(pretrained=True)`: Loads the R3D-18 model with pre-trained weights (likely from Kinetics-400). This leverages knowledge learned from a large video dataset.
    - `model.fc = nn.Linear(model.fc.in_features, 2)`: The original R3D-18 model is designed for a different number of classes (400 for Kinetics). This line replaces the final fully connected layer (`fc`) with a new linear layer that outputs 2 values, corresponding to the two classes in this binary classification task (violent/non-violent).
    - `model.to(device)`: Moves the model to the selected device (GPU or CPU).
- **Loss Function:**
    - `nn.CrossEntropyLoss()`: This is the standard loss function for multi-class classification problems. It calculates the cross-entropy between the predicted probabilities and the true labels.
- **Optimizer:**
    - `optim.Adam(model.parameters(), lr=0.001)`: The Adam optimizer is chosen to update the model's weights during training. The learning rate (`lr`) is initially set to 0.001 and can be adjusted. The optimizer is configured to optimize all the parameters of the model.

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pre-trained 3D ResNet and modify for 2 classes
model = r3d_18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)  # Output for violent/non-violent
model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adjust LR as needed

/home/heytanix/Documents/Jain_PCL/PCL_repository/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/heytanix/Documents/Jain_PCL/PCL_repository/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /home/heytanix/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100.0%


### Custom Video Transformations (Resize and Normalize)

This cell defines custom transformation classes `ResizeVideo` and `NormalizeVideo` that are specifically designed to work with video tensors (which have a time dimension). While `torchvision.transforms` primarily works on single images, these classes apply the transformations consistently across all frames of a video clip.

- **`ResizeVideo`**:
    - Takes a target size as input.
    - The `__call__` method iterates through each frame of the input video tensor (`video: (C, T, H, W)`).
    - For each frame, it uses `torch.nn.functional.interpolate` with `mode='bilinear'` to resize the frame to the target size.
    - The resized frames are collected into a new video tensor.

- **`NormalizeVideo`**:
    - Takes mean and standard deviation values as input.
    - It reshapes the mean and standard deviation tensors to have a shape `(C, 1, 1)` so they can be correctly broadcasted across the time, height, and width dimensions when performing the normalization.
    - The `__call__` method applies the standard normalization formula `(video - mean) / std` to the entire video tensor.

Finally, an updated `transforms.Compose` object is created using these custom classes. This composition will be used when creating the `VideoDataset` instances to preprocess the video data.

In [ ]:
# Custom resize class for videos (applies to each frame)
class ResizeVideo:
    def __init__(self, size):
        self.size = size

    def __call__(self, video):
        # video: (C, T, H, W)
        c, t, h, w = video.shape
        video_resized = torch.zeros((c, t, self.size, self.size), dtype=video.dtype)
        for i in range(t):
            frame = video[:, i, :, :].unsqueeze(0)  # (1, C, H, W)
            frame_resized = F.interpolate(frame, size=(self.size, self.size), mode='bilinear', align_corners=False)
            video_resized[:, i, :, :] = frame_resized.squeeze(0)
        return video_resized

# Custom normalize class for videos (applies to each frame)
class NormalizeVideo:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean).view(-1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1)

    def __call__(self, video):
        # video: (C, T, H, W)
        return (video - self.mean[..., None, None]) / self.std[..., None, None]

# Updated transforms composition
transform = transforms.Compose([
    ResizeVideo(112),  # Resize each frame to 112x112
    NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])  # Normalize per channel across frames
])

### Corrected Custom Video Transformations

This cell provides a corrected version of the custom `NormalizeVideo` class.

- **`ResizeVideo`**: Remains the same as the previous cell, correctly resizing each frame individually.

- **`NormalizeVideo` (Corrected)**:
    - Takes mean and standard deviation values.
    - It reshapes the mean and standard deviation tensors to have a shape `(C, 1, 1, 1)`. This shape is crucial for correctly broadcasting the mean and standard deviation across the time (T), height (H), and width (W) dimensions of the video tensor `(C, T, H, W)`. This ensures that normalization is applied channel-wise across all frames and pixels.
    - The `__call__` method applies the normalization using the correctly shaped mean and standard deviation.

Finally, the `transforms.Compose` is updated to use the corrected `NormalizeVideo` class.

In [ ]:
# Custom resize for videos (resizes each frame individually)
class ResizeVideo:
    def __init__(self, size):
        self.size = size

    def __call__(self, video):
        # video: (C, T, H, W)
        c, t, h, w = video.shape
        video_resized = torch.zeros((c, t, self.size, self.size), dtype=video.dtype)
        for i in range(t):
            frame = video[:, i, :, :].unsqueeze(0)  # (1, C, H, W)
            frame_resized = F.interpolate(frame, size=(self.size, self.size), mode='bilinear', align_corners=False)
            video_resized[:, i, :, :] = frame_resized.squeeze(0)
        return video_resized

# Custom normalize for videos (broadcasts mean/std across T, H, W)
class NormalizeVideo:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean).view(-1, 1, 1, 1)  # Shape for broadcasting: (C, 1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1, 1)

    def __call__(self, video):
        # video: (C, T, H, W)
        return (video - self.mean) / self.std

# Updated transforms
transform = transforms.Compose([
    ResizeVideo(112),  # Resize to 112x112 per frame
    NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])  # Normalize across all frames
])

### Save Model State Dictionary

This cell saves the current state of the neural network model's parameters (weights and biases) to a file.

- `torch.save(model.state_dict(), 'violence_classifier.pth')`: This line uses `torch.save` to serialize the model's `state_dict()` (a Python dictionary containing all the parameter tensors) and saves it to a file named `violence_classifier.pth`. Saving the state dictionary allows you to load the trained model later without needing to redefine the model architecture.
- A success message is printed to confirm that the model state was saved.

In [ ]:
torch.save(model.state_dict(), 'violence_classifier.pth')
print("Model saved successfully!")